In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

base_path = Path().resolve().parent
file_path = base_path / 'data' / 'raw' / 'Stellar_edu_MDS_ap_stats_dataset - v1.9.xlsx'

In [3]:
data = pd.read_excel(file_path, sheet_name='Student_Observations')

In [4]:
data.head()

,student_id,assignment_id,class_num,observation_id,item_type,source_question,primary_kc_id,all_kc_ids,score,max_score,percent_score,student_response,correct_answer_or_rubric,rubric_level
0,S001,HW1,1,HW1_PCA_Q01,MCQ,PCA Q01,KC.U1.02.observational_unit_variable,KC.U1.02.observational_unit_variable,0.0,1,0,C,E,NaN
1,S001,HW1,1,HW1_PCA_Q02,MCQ,PCA Q02,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,1.0,1,100,E,E,NaN
2,S001,HW1,1,HW1_PCA_Q03,MCQ,PCA Q03,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,0.0,1,0,C,D,NaN
3,S001,HW1,1,HW1_PCA_Q04,MCQ,PCA Q04,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,E,E,NaN
4,S001,HW1,1,HW1_PCA_Q05,MCQ,PCA Q05,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,B,B,NaN


In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 21808 entries, 0 to 21807
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   student_id                21808 non-null  str    
 1   assignment_id             21808 non-null  str    
 2   class_num                 21808 non-null  int64  
 3   observation_id            21808 non-null  str    
 4   item_type                 21808 non-null  str    
 5   source_question           21808 non-null  str    
 6   primary_kc_id             21808 non-null  str    
 7   all_kc_ids                21808 non-null  str    
 8   score                     21808 non-null  float64
 9   max_score                 21808 non-null  int64  
 10  percent_score             21808 non-null  int64  
 11  student_response          21808 non-null  str    
 12  correct_answer_or_rubric  21808 non-null  str    
 13  rubric_level              4642 non-null   str    
dtypes: float64(1), in

In [6]:
data.nunique()

student_id                   25
assignment_id                33
class_num                    27
observation_id              895
item_type                     3
source_question             640
primary_kc_id               236
all_kc_ids                  521
score                         3
max_score                     1
percent_score                 3
student_response             25
correct_answer_or_rubric    185
rubric_level                  3
dtype: int64

**The Criteria for the "Best Subset"**
To make BKT work, a Knowledge Component (KC) must meet three strict criteria:

- Vertical Depth (Sequence): Students must have attempted the skill at least 3+ times on average. BKT cannot "see" learning if there is only one data point per person.

- Horizontal Breadth (Reach): At least 5–10 students should have attempted the skill. If only one student did a KC, the model will "overfit" to that one person's specific mistakes.

- Statistical Variance: The KC cannot have 100% accuracy or 0% accuracy. The model needs to see the "flip" from incorrect to correct to calculate the Learn Rate.

In [7]:
data.columns

Index(['student_id', 'assignment_id', 'class_num', 'observation_id',
       'item_type', 'source_question', 'primary_kc_id', 'all_kc_ids', 'score',
       'max_score', 'percent_score', 'student_response',
       'correct_answer_or_rubric', 'rubric_level'],
      dtype='str')

In [15]:
kc_stats = data.groupby('primary_kc_id').agg(
    total_no_obs=('score', 'count'),
    unique_students=('student_id', 'nunique'),
    avg_score= ('score', 'mean'),
    avg_std=('score', 'std')
)

kc_stats.sort_values(by='total_no_obs')

,total_no_obs,unique_students,avg_score,avg_std
primary_kc_id,,,,
KC.U8.16.homogeneity_hypotheses,23,23,0.608696,0.499011
KC.U10.03.bootstrap_same_sample_size,23,23,0.739130,0.448978
KC.U7.09.generalization_scope_quantitative_inference,23,23,0.586957,0.325163
KC.U7.03.standard_error_mean,23,23,0.608696,0.499011
KC.U6.25.two_prop_ci_standard_error,23,23,0.652174,0.486985
...,...,...,...,...
KC.U4.01.probability_long_run_interpretation,224,25,0.633929,0.482808
KC.U2.03.conditional_relative_frequency,225,25,0.577778,0.483610
KC.U4.11.independent_events_probability,324,25,0.561728,0.496942


In [14]:
kc_stats.describe()

,total_no_obs,unique_students,avg_score,avg_std
count,236.000000,236.000000,236.000000,236.000000
mean,92.406780,24.699153,0.527557,0.479459
std,58.257132,0.651152,0.079650,0.029188
min,23.000000,23.000000,0.315217,0.325163
25%,48.750000,25.000000,0.478261,0.467363
50%,74.000000,25.000000,0.522468,0.489898
75%,124.000000,25.000000,0.575086,0.501285
max,423.000000,25.000000,0.740000,0.510754


In [16]:
kc_stats.describe()

,total_no_obs,unique_students,avg_score,avg_std
count,236.000000,236.000000,236.000000,236.000000
mean,92.406780,24.699153,0.527557,0.479459
std,58.257132,0.651152,0.079650,0.029188
min,23.000000,23.000000,0.315217,0.325163
25%,48.750000,25.000000,0.478261,0.467363
50%,74.000000,25.000000,0.522468,0.489898
75%,124.000000,25.000000,0.575086,0.501285
max,423.000000,25.000000,0.740000,0.510754
